# create Raster from CSV containing data about SSA countries

In [8]:
import pandas as pd
import numpy as np
import rasterio
import geopandas as gpd
from rasterio.features import rasterize
from rasterio.transform import from_bounds
from pathlib import Path
import urllib.request

"""
This script converts a CSV file containing country-level data (csv_path) into a multi-band GeoTIFF raster (output_path). Each band corresponds to a selected column (selected_columns) from the CSV, 
mapped onto national polygons. Missing values for Sub-Saharan African (SSA) countries can be filled with a weighted average based on pixel counts (fill_missing_with_average = True/False)
The output raster is georeferenced and includes metadata for each band.
"""

# --- CONFIGURATION ---
csv_path = r"C:/Users/Fra/OneDrive - Politecnico di Milano/File di Mattia Cinchetti - Thesis_onstoveM&F/OnStoveThesis/thesis/script_nostri/dataset_first_step/lpg_household_shares.csv"
output_path = r"C:/Users/Fra/OneDrive - Politecnico di Milano/File di Mattia Cinchetti - Thesis_onstoveM&F/OnStoveThesis/thesis/script_nostri/dataset_first_step/lpg_household_shares.tif"
csv_iso_column = "country" 
selected_columns = {
    "percentage_house": "percentage_house" 
}
fill_missing_with_average = True 
NODATA_VALUE = -9999.0  # Defined globally for consistency

# Default Global Grid
bounds = (-180, -90, 180, 90) 
resolution = 0.1 

ssa_iso3 = [
    'AGO', 'BEN', 'BWA', 'BFA', 'BDI', 'CPV', 'CMR', 'CAF', 'TCD', 'COM',
    'COD', 'COG', 'CIV', 'DJI', 'GNQ', 'ERI', 'SWZ', 'ETH', 'GAB', 'GMB',
    'GHA', 'GIN', 'GNB', 'KEN', 'LSO', 'LBR', 'MDG', 'MWI', 'MLI', 'MRT',
    'MUS', 'MOZ', 'NAM', 'NER', 'NGA', 'RWA', 'STP', 'SEN', 'SYC', 'SLE',
    'SOM', 'ZAF', 'SSD', 'SDN', 'TZA', 'TGO', 'UGA', 'ZMB', 'ZWE',
]

# --- STEP 0: LOAD BOUNDARIES ---
print("Loading national boundaries...")
cache_dir = Path.home() / ".geopandas_cache"
cache_dir.mkdir(exist_ok=True)
ne_zip = cache_dir / "ne_10m_admin_0_countries.zip"

if not ne_zip.exists():
    ne_url = "https://naciscdn.org/naturalearth/10m/cultural/ne_10m_admin_0_countries.zip"
    urllib.request.urlretrieve(ne_url, ne_zip)

world = gpd.read_file(ne_zip).to_crs("EPSG:4326")

iso_candidates = ["ISO_A3", "ADM0_A3", "SOV_A3", "GU_A3"]
iso_col = next((c for c in iso_candidates if c in world.columns), None)
world["iso3"] = world[iso_col].astype(str).str.strip().str.upper()

# Define raster dimensions
width = int((bounds[2] - bounds[0]) / resolution)
height = int((bounds[3] - bounds[1]) / resolution)
transform = from_bounds(*bounds, width, height)

# Pre-calculate pixel counts per country
print("Calculating pixel counts per country...")
world['pixel_count'] = 0
for idx, row in world.iterrows():
    if row["geometry"] is not None:
        mask = rasterize(
            [(row["geometry"], 1)],
            out_shape=(height, width),
            transform=transform,
            fill=0,
            dtype=np.uint8,
            all_touched=True
        )
        world.at[idx, 'pixel_count'] = mask.sum()

# --- STEP 1: LOAD CSV ---
print(f"Loading data from: {csv_path}")
df = pd.read_csv(csv_path)
df[csv_iso_column] = df[csv_iso_column].astype(str).str.strip().str.upper()

if selected_columns:
    missing_cols = [col for col in selected_columns if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Selected columns not found in CSV: {missing_cols}")
    value_columns = list(selected_columns.keys())
    band_names = [selected_columns[col] for col in value_columns]
else:
    value_columns = [col for col in df.columns if col != csv_iso_column]
    band_names = value_columns.copy()

for col in value_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# --- STEP 2: MAP DATA ---
print("Mapping data onto national polygons...")
df_indexed = df.set_index(csv_iso_column)
for col in value_columns:
    world[f"val_{col}"] = world["iso3"].map(df_indexed[col])

# --- STEP 3: RASTERIZATION ---
print(f"Creating raster with {len(value_columns)} bands...")
with rasterio.open(
    output_path, "w",
    driver="GTiff",
    height=height, width=width,
    count=len(value_columns),
    dtype=np.float32,
    crs="EPSG:4326",
    transform=transform,
    nodata=NODATA_VALUE  # Sets the NoData value in the TIFF metadata
) as dst:
    
    for band_idx, (col, band_name) in enumerate(zip(value_columns, band_names), start=1):
        target_col = f"val_{col}"
        
        # SSA Filling Logic
        if fill_missing_with_average:
            is_ssa = world["iso3"].isin(ssa_iso3)
            valid_ssa = world[is_ssa & world[target_col].notna()]
            missing_ssa = world[is_ssa & world[target_col].isna()]
            
            if not valid_ssa.empty and not missing_ssa.empty:
                total_valid_pixels = valid_ssa['pixel_count'].sum()
                if total_valid_pixels > 0:
                    weighted_sum = (valid_ssa[target_col] * valid_ssa['pixel_count']).sum()
                    weighted_avg = weighted_sum / total_valid_pixels
                    world.loc[missing_ssa.index, target_col] = weighted_avg
                    print(f" - Band {band_idx}: Filled {len(missing_ssa)} SSA countries with average ({weighted_avg:.4f})")
        
        # Generate the shapes to rasterize (skipping NaN values)
        shapes = [
            (row["geometry"], float(row[target_col])) 
            for _, row in world.iterrows()
            if pd.notna(row[target_col]) and row["geometry"] is not None
        ]

        # Rasterize shapes. 'fill' handles the background/missing areas.
        raster_array = rasterize(
            shapes,
            out_shape=(height, width),
            transform=transform,
            fill=NODATA_VALUE, # Consistent with the TIFF metadata
            dtype=np.float32,
            all_touched=True
        )
        
        # Write band and update metadata tags
        dst.write(raster_array, band_idx)
        dst.update_tags(band_idx, long_name=band_name)
        print(f" - Band {band_idx} ({band_name}) saved.")

print(f"\nProcessing complete. File: {output_path}")

Loading national boundaries...
Calculating pixel counts per country...
Loading data from: C:/Users/Fra/OneDrive - Politecnico di Milano/File di Mattia Cinchetti - Thesis_onstoveM&F/OnStoveThesis/thesis/script_nostri/dataset_first_step/lpg_household_shares.csv
Mapping data onto national polygons...
Creating raster with 1 bands...
 - Band 1: Filled 2 SSA countries with average (75.9118)
 - Band 1 (percentage_house) saved.

Processing complete. File: C:/Users/Fra/OneDrive - Politecnico di Milano/File di Mattia Cinchetti - Thesis_onstoveM&F/OnStoveThesis/thesis/script_nostri/dataset_first_step/lpg_household_shares.tif
